# A tour of NoisiQ

Build a circuit, attach noise, simulate, and *see* where errors happen and how they spread. Run the cells top to bottom; every figure is generated live.

In [ ]:
import numpy as np

import noisiq as nq

%matplotlib inline

## 1. Circuits
Use the fluent builder, the algorithm library, or import OpenQASM/Qiskit. Any unitary is allowed.

In [ ]:
c = nq.Circuit(4, name="demo").h(0).cx(0, 1).rz(np.pi / 3, 1).ccx(0, 1, 2).swap(2, 3)
c.measure_all()
c.draw();

In [ ]:
nq.algorithms.grover(3, "101").draw();

## 2. Watching a single error
`nq.propagate` injects a Pauli error and tracks it exactly through Clifford gates.

In [ ]:
trace = nq.propagate(nq.algorithms.ghz(5, uncompute=True), "X", qubit=2, after=2)
print(trace.pauli_strings())
nq.plot_propagation(trace);

In [ ]:
from IPython.display import HTML

HTML(nq.animate_propagation(trace).to_jshtml())

## 3. Noise models
Rules attach noise to gates, idle periods and measurements. `nq.explain` prints the math.

In [ ]:
noise = nq.NoiseModel("demo noise")
noise.add_gate_error(nq.noise.depolarizing(0.002), num_qubits=1)
noise.add_gate_error(nq.noise.depolarizing(0.02, 2), num_qubits=2)
noise.add_thermal_relaxation(t1=100e-6, t2=80e-6)
noise.add_readout_error(0.01)
noise

In [ ]:
nq.explain("thermal_relaxation");

## 4. Simulate and look
Clifford circuits with Pauli noise use the fast frame backend; everything else the statevector backend.

In [ ]:
code = nq.algorithms.repetition_code(3, rounds=1)
result = nq.simulate(code, noise.twirled(), shots=5000, seed=1, trace_shots=20)
print(result.summary())
nq.plot_error_heatmap(result, annotate=True);

In [ ]:
nq.plot_error_budget(result, metric="burden");

In [ ]:
nq.plot_propagation(result);

## 5. Real hardware profiles

In [ ]:
qft = nq.algorithms.qft(4, input_state="0101")
r = nq.simulate(qft, nq.get_hardware("ibm_eagle_r3").noise_model(), shots=300, seed=1)
print(r.fidelity)
nq.plot_bloch(r);

## 6. Dynamical decoupling
Slow frequency drift is refocused by echo pulses.

In [ ]:
ramsey = nq.Circuit(1).h(0).delay(0, 20e-6).h(0).measure(0)
drift = nq.NoiseModel("drift").add_quasistatic_detuning(2 * np.pi * 30e3)
runs = {
    name: nq.simulate(circ, drift, shots=2000, seed=1)
    for name, circ in [("no DD", ramsey), ("XY4", nq.suppression.apply_dd(ramsey, "XY4"))]
}
nq.plot_comparison(runs, "fidelity");